# 🎬 ASTRAA — LTX-Video Colab
Free/open-source image-to-video test for ASTRAA.

## 1. Check GPU
Run this first. A free T4 is ideal for this lightweight test.

In [ ]:
!nvidia-smi


## 2. Install LTX-Video
Uses the official Lightricks repository.

In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git
%cd /content/LTX-Video
!pip install -q -e '.[inference]'
!pip install -q --force-reinstall --no-deps 'huggingface-hub~=0.30'


## 3. Download the lighter 2B distilled model
This is intended for lighter VRAM than the 13B model.

In [ ]:
from huggingface_hub import hf_hub_download
model_dir='/content/LTX-Video/models'
import os
os.makedirs(model_dir, exist_ok=True)
hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=model_dir)
print('Model downloaded:', model_dir)


## 4. Upload the ASTRAA reference image
Upload one image such as Aarav + Maa Meera. Keep the image in `/content/LTX-Video/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
image_name=next(iter(uploaded))
image_path=f'/content/LTX-Video/{image_name}'
print(image_path)


## 5. Generate the first test shot
Start with a short 49-frame test first. The current LTX-Video 0.9.8 config supplies the checkpoint itself, so we do not pass `--checkpoint_path` separately.


In [ ]:
import os
%cd /content/LTX-Video
!git fetch origin main --depth 1
!git checkout FETCH_HEAD -- inference.py configs/ltxv-2b-0.9.8-distilled.yaml
!python /content/LTX-Video/inference.py --prompt "$PROMPT" --conditioning_media_paths "$image_path" --conditioning_start_frames 0 --height 512 --width 768 --num_frames 49 --seed 42 --pipeline_config /content/LTX-Video/configs/ltxv-2b-0.9.8-distilled.yaml


In [ ]:
import os, glob
videos=glob.glob('/content/LTX-Video/**/*.mp4', recursive=True)
print('\n'.join(videos[-10:]) if videos else 'No MP4 found yet.')


## Next
Once the first shot works, we will add reusable ASTRAA prompts and an extension workflow.